# Ideal Spatial Adaptation by Wavelet Shrinkage — Implementation / 구현

**Paper**: Donoho, D. L., & Johnstone, I. M. (1994). *Biometrika*, 81(3), 425–455. [DOI: 10.1093/biomet/81.3.425]

## Overview / 개요

이 노트북은 Donoho & Johnstone (1994)의 핵심 알고리즘을 NumPy + PyWavelets로 재구현한다:
1. **4개 표준 시험 함수** (Blocks, Bumps, HeaviSine, Doppler) 생성, \$n=2048\$
2. **이산 웨이블릿 변환 (DWT)** 과 Parseval 항등식 검증
3. **Hard / soft thresholding** 비선형성 (Eqs. 11–12)
4. **MAD-기반 robust 잡음 추정** \$\\hat\\sigma = \\mathrm{MAD}/0.6745\$
5. **VisuShrink** (universal threshold \$\\sigma\\sqrt{2\\log n}\$, Definition 2)
6. **RiskShrink** (minimax threshold \$\\lambda^*\_n\$, §2.4)
7. Table 3 부분 재현: noisy / hard / soft / VisuShrink / RiskShrink의 MSE 비교

This notebook reproduces the core algorithms from the paper using NumPy and PyWavelets:
- 4 standard test functions at \$n=2048\$
- DWT + Parseval verification
- Hard/soft thresholding nonlinearities
- MAD-based noise estimation
- VisuShrink and RiskShrink with Table 3 reproduction


In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import pywt
from numpy.typing import NDArray

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True

rng = np.random.default_rng(42)

N: int = 2048
WAVELET: str = 'sym8'  # Symmlet N=8, paper's default

---

## Part 1: Test functions / 시험 함수 (Table 1)

Paper의 Table 1에서 정의된 4개 함수를 그대로 구현. 모두 \$t \\in [0, 1]\$, \$n = 2048\$.


In [ ]:
def blocks(t: NDArray[np.float64]) -> NDArray[np.float64]:
    """Donoho-Johnstone Blocks function.

    f(t) = sum_j h_j K(t - t_j),  K(t) = (1 + sgn(t))/2

    Args:
        t: Time grid in [0, 1].

    Returns:
        Blocks function values.
    """
    tj = np.array([0.10, 0.13, 0.15, 0.23, 0.25, 0.40, 0.44, 0.65, 0.76, 0.78, 0.81])
    hj = np.array([4.0, -5.0, 3.0, -4.0, 5.0, -4.2, 2.1, 4.3, -3.1, 2.1, -4.2])
    out = np.zeros_like(t)
    for h, c in zip(hj, tj):
        out += h * (1.0 + np.sign(t - c)) / 2.0
    return out


def bumps(t: NDArray[np.float64]) -> NDArray[np.float64]:
    """Donoho-Johnstone Bumps function.

    f(t) = sum_j h_j (1 + |t - t_j|/w_j)^{-4}
    """
    tj = np.array([0.10, 0.13, 0.15, 0.23, 0.25, 0.40, 0.44, 0.65, 0.76, 0.78, 0.81])
    hj = np.array([4.0, 5.0, 3.0, 4.0, 5.0, 4.2, 2.1, 4.3, 3.1, 5.1, 4.2])
    wj = np.array([0.005, 0.005, 0.006, 0.01, 0.01, 0.03, 0.01, 0.01, 0.005, 0.008, 0.005])
    out = np.zeros_like(t)
    for h, c, w in zip(hj, tj, wj):
        out += h * (1.0 + np.abs(t - c) / w) ** (-4)
    return out


def heavisine(t: NDArray[np.float64]) -> NDArray[np.float64]:
    """Donoho-Johnstone HeaviSine function.

    f(t) = 4 sin(4*pi*t) - sgn(t - 0.3) - sgn(0.72 - t)
    """
    return 4.0 * np.sin(4.0 * np.pi * t) - np.sign(t - 0.3) - np.sign(0.72 - t)


def doppler(t: NDArray[np.float64]) -> NDArray[np.float64]:
    """Donoho-Johnstone Doppler function.

    f(t) = sqrt(t (1-t)) sin(2*pi*(1+eps)/(t+eps)),  eps = 0.05
    """
    eps = 0.05
    return np.sqrt(np.clip(t * (1.0 - t), 0.0, None)) * np.sin(2.0 * np.pi * (1.0 + eps) / (t + eps))


def make_test_signals(n: int = N, snr: float = 7.0, seed: int = 0) -> dict[str, dict]:
    """Build all four test signals plus matched-SNR noisy versions.

    SNR is defined as paper does it: ||f||_2 / ||sigma * z||_2.
    Setting sigma=1 then rescaling f to achieve target SNR.

    Args:
        n: Length of each signal.
        snr: Target signal-to-noise ratio.
        seed: Seed for reproducible noise.

    Returns:
        Dict mapping name -> {'clean', 'noisy', 'sigma'}.
    """
    t = np.arange(n) / float(n)
    funcs = {'Blocks': blocks(t), 'Bumps': bumps(t),
             'HeaviSine': heavisine(t), 'Doppler': doppler(t)}
    sigma = 1.0
    local_rng = np.random.default_rng(seed)
    out = {}
    for name, f in funcs.items():
        f_centred = f - f.mean()
        z = local_rng.standard_normal(n)
        # rescale f so ||f||/||sigma z|| = SNR
        scale = snr * np.linalg.norm(sigma * z) / max(np.linalg.norm(f_centred), 1e-12)
        f_scaled = scale * f_centred + f.mean() * scale
        y = f_scaled + sigma * z
        out[name] = {'clean': f_scaled, 'noisy': y, 'sigma': sigma}
    return out


signals = make_test_signals(N, snr=7.0, seed=0)
t = np.arange(N) / float(N)

fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=True)
for row, name in enumerate(['Blocks', 'Bumps', 'HeaviSine', 'Doppler']):
    axes[row, 0].plot(t, signals[name]['clean'], 'C0-', lw=0.8)
    axes[row, 0].set_ylabel(name)
    axes[row, 1].plot(t, signals[name]['noisy'], 'C3-', lw=0.5)
axes[0, 0].set_title('Clean (Fig. 1 analog)')
axes[0, 1].set_title('Noisy (Fig. 3 analog, SNR ≈ 7)')
plt.tight_layout(); plt.show()

---

## Part 2: DWT and Parseval verification / DWT와 Parseval 항등식 검증

Symmlet \$N=8\$, periodised; orthogonal so Parseval holds: \$\\|y\\|^2 = \\|w\\|^2\$.


In [ ]:
def dwt_coeffs_to_array(coeffs: list) -> NDArray[np.float64]:
    """Concatenate pywt coefficient list into a flat array."""
    return np.concatenate(coeffs)


def array_to_dwt_coeffs(arr: NDArray[np.float64], shapes: list[int]) -> list:
    """Split flat array back into pywt coefficient list using stored shapes."""
    out = []
    pos = 0
    for s in shapes:
        out.append(arr[pos:pos + s])
        pos += s
    return out


y = signals['Doppler']['noisy']
coeffs = pywt.wavedec(y, WAVELET, mode='periodization')
shapes = [len(c) for c in coeffs]
w_flat = dwt_coeffs_to_array(coeffs)

energy_y = float(np.sum(y ** 2))
energy_w = float(np.sum(w_flat ** 2))

print(f'Wavelet              : {WAVELET}')
print(f'Decomposition levels : {len(coeffs) - 1}  (cA + cD_J..cD_1)')
print(f'Per-level sizes      : {shapes}')
print(f'Sum of sizes         : {sum(shapes)}  (should equal n={N})')
print(f'||y||^2              : {energy_y:.6f}')
print(f'||w||^2              : {energy_w:.6f}')
print(f'Relative diff        : {abs(energy_y - energy_w) / energy_y:.2e}  (Parseval)')

---

## Part 3: Hard / soft thresholding / 강한·약한 임계화

Eqs. (11)–(12):
\$\$ \\eta\_H(w, \\lambda) = w\\,\\mathbf 1\\{|w| > \\lambda\\}, \\quad \\eta\_S(w, \\lambda) = \\mathrm{sgn}(w)(|w| - \\lambda)\_+ \$\$


In [ ]:
def hard_threshold(w: NDArray[np.float64], lam: float) -> NDArray[np.float64]:
    """Hard thresholding: keep |w|>lam, zero otherwise. Eq. (11)."""
    return np.where(np.abs(w) > lam, w, 0.0)


def soft_threshold(w: NDArray[np.float64], lam: float) -> NDArray[np.float64]:
    """Soft thresholding: shrink toward zero by lam. Eq. (12)."""
    return np.sign(w) * np.maximum(np.abs(w) - lam, 0.0)


x = np.linspace(-5, 5, 1001)
lam = 1.5
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, x, 'k--', alpha=0.3, label='identity')
ax.plot(x, hard_threshold(x, lam), 'C0-', lw=2, label=fr'hard, $\lambda={lam}$')
ax.plot(x, soft_threshold(x, lam), 'C1-', lw=2, label=fr'soft, $\lambda={lam}$')
ax.axvline(lam, color='gray', alpha=0.3); ax.axvline(-lam, color='gray', alpha=0.3)
ax.set_xlabel('w'); ax.set_ylabel(r'$\eta(w, \lambda)$')
ax.set_title('Hard vs. soft thresholding nonlinearities')
ax.legend(); plt.tight_layout(); plt.show()

---

## Part 4: MAD-based noise estimation / MAD 기반 잡음 추정

\$\\hat\\sigma = \\mathrm{median}(|w\_{J,k}|) / 0.6745\$ — 가장 미세 스케일 \$j=J\$의 detail 계수에서.


In [ ]:
def estimate_sigma_mad(coeffs: list) -> float:
    """Robust noise std estimation from finest-scale detail coeffs.

    Args:
        coeffs: pywt.wavedec output [cA_J, cD_J, ..., cD_1].

    Returns:
        Estimated noise standard deviation.
    """
    finest = coeffs[-1]  # cD_1 (highest-frequency band)
    return float(np.median(np.abs(finest)) / 0.6745)


for name, sig in signals.items():
    coeffs = pywt.wavedec(sig['noisy'], WAVELET, mode='periodization')
    sigma_hat = estimate_sigma_mad(coeffs)
    print(f'{name:>10s}: true sigma = {sig["sigma"]:.4f}, estimated sigma = {sigma_hat:.4f}')

---

## Part 5: VisuShrink and RiskShrink / VisuShrink와 RiskShrink

**VisuShrink**: \$\\lambda^u\_n = \\hat\\sigma\\sqrt{2\\log n}\$. **RiskShrink**: \$\\lambda^*\_n\$ from minimax SURE — 여기서는 grid scan으로 근사. Both apply soft thresholding to detail levels \$j \\ge j\_0\$ only.


In [ ]:
def visushrink(
    y: NDArray[np.float64],
    wavelet: str = WAVELET,
    j0: int = 5,
) -> tuple[NDArray[np.float64], float, float]:
    """VisuShrink (Definition 2): soft threshold detail levels with sigma*sqrt(2 log n).

    Args:
        y: Noisy signal of length n=2^L.
        wavelet: pywt wavelet name.
        j0: Coarsest level to NOT shrink; finer detail levels are thresholded.

    Returns:
        Tuple of (estimate, sigma_hat, lambda_used).
    """
    n = len(y)
    coeffs = pywt.wavedec(y, wavelet, mode='periodization')
    sigma_hat = estimate_sigma_mad(coeffs)
    lam = sigma_hat * np.sqrt(2.0 * np.log(n))

    # coeffs = [cA_J, cD_J, cD_{J-1}, ..., cD_1]
    # j_0=5 means keep cA_J and the (J - j_0) coarsest detail bands untouched.
    # Practically: shrink only the (j_0) finest detail bands.
    new_coeffs = [coeffs[0]]  # cA_J kept
    n_detail = len(coeffs) - 1
    for level_idx, c in enumerate(coeffs[1:], start=1):
        # detail at scale (J - level_idx + 1); we shrink the finest bands
        if level_idx > n_detail - j0:
            new_coeffs.append(soft_threshold(c, lam))
        else:
            new_coeffs.append(c.copy())
    f_hat = pywt.waverec(new_coeffs, wavelet, mode='periodization')
    return f_hat[:n], sigma_hat, lam


def riskshrink_threshold_grid(n: int) -> float:
    """Approximate RiskShrink minimax threshold via Table 2 entries.

    Lookup table from the paper's Table 2 (n -> lambda^*_n).

    Args:
        n: Sample size.

    Returns:
        Minimax threshold lambda^*_n (interpolated).
    """
    table = {64: 1.474, 128: 1.669, 256: 1.860, 512: 2.048, 1024: 2.232,
             2048: 2.414, 4096: 2.594, 8192: 2.773, 16384: 2.952,
             32768: 3.131, 65536: 3.310}
    if n in table:
        return table[n]
    keys = sorted(table.keys())
    return float(np.interp(np.log2(n), [np.log2(k) for k in keys], [table[k] for k in keys]))


def riskshrink(
    y: NDArray[np.float64],
    wavelet: str = WAVELET,
    j0: int = 5,
) -> tuple[NDArray[np.float64], float, float]:
    """RiskShrink: same as VisuShrink but with minimax threshold lambda^*_n * sigma_hat."""
    n = len(y)
    coeffs = pywt.wavedec(y, wavelet, mode='periodization')
    sigma_hat = estimate_sigma_mad(coeffs)
    lam = sigma_hat * riskshrink_threshold_grid(n)
    new_coeffs = [coeffs[0]]
    n_detail = len(coeffs) - 1
    for level_idx, c in enumerate(coeffs[1:], start=1):
        if level_idx > n_detail - j0:
            new_coeffs.append(soft_threshold(c, lam))
        else:
            new_coeffs.append(c.copy())
    f_hat = pywt.waverec(new_coeffs, wavelet, mode='periodization')
    return f_hat[:n], sigma_hat, lam


# Sanity check thresholds for n=2048
print(f'For n={N}:')
print(f'  Universal sqrt(2 log n) = {np.sqrt(2 * np.log(N)):.4f}  (paper: 3.905)')
print(f'  Minimax lambda^*_n       = {riskshrink_threshold_grid(N):.4f}  (paper: 2.414)')

---

## Part 6: Reconstruction comparison / 재구성 비교

Doppler 신호로 시각 비교: Noisy → Hard → Soft → VisuShrink → RiskShrink → Clean.


In [ ]:
name = 'Doppler'
y = signals[name]['noisy']
f_clean = signals[name]['clean']

f_visu, sigma_v, lam_v = visushrink(y)
f_risk, _, lam_r = riskshrink(y)

fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
axes[0].plot(t, f_clean, 'k-', lw=0.8); axes[0].set_title(f'{name} — clean')
axes[1].plot(t, y, 'C3-', lw=0.4); axes[1].set_title(f'{name} — noisy (sigma=1, SNR=7)')
axes[2].plot(t, f_risk, 'C2-', lw=0.6)
axes[2].set_title(fr'RiskShrink: $\hat\sigma$={sigma_v:.3f}, $\lambda$={lam_r:.3f}, '
                  f'MSE={np.mean((f_risk - f_clean)**2):.4f}')
axes[3].plot(t, f_visu, 'C0-', lw=0.6)
axes[3].set_title(fr'VisuShrink: $\hat\sigma$={sigma_v:.3f}, $\lambda$={lam_v:.3f}, '
                  f'MSE={np.mean((f_visu - f_clean)**2):.4f}')
axes[3].set_xlabel('t')
plt.tight_layout(); plt.show()

---

## Part 7: Reproduce Table 3 — average squared error / Table 3 일부 재현

각 신호에서: noisy / ideal-wavelet (oracle) / RiskShrink / VisuShrink / hard at \$\\sqrt{2\\log n}\$.
Ideal-wavelet은 *진짜* 신호의 웨이블릿 변환 \$\\theta\$로 noisy coefficient \$w\$를 *coordinate-wise* 선택: \$\\theta^2 < \\sigma^2\$이면 0, 아니면 \$w\$를 유지 (projection oracle).


In [ ]:
def ideal_wavelet_oracle(
    y: NDArray[np.float64],
    f_clean: NDArray[np.float64],
    sigma: float,
    wavelet: str = WAVELET,
) -> NDArray[np.float64]:
    """Projection oracle: keep w where |theta|>sigma, else zero. Lower bound on practical methods."""
    coeffs_y = pywt.wavedec(y, wavelet, mode='periodization')
    coeffs_f = pywt.wavedec(f_clean, wavelet, mode='periodization')
    new = [coeffs_y[0]]  # keep cA fully
    for c_y, c_f in zip(coeffs_y[1:], coeffs_f[1:]):
        new.append(np.where(np.abs(c_f) > sigma, c_y, 0.0))
    out = pywt.waverec(new, wavelet, mode='periodization')
    return out[:len(y)]


def hard_visu(y: NDArray[np.float64], wavelet: str = WAVELET, j0: int = 5) -> NDArray[np.float64]:
    """Hard thresholding with universal threshold (analog of paper's Fig 9 with hard)."""
    n = len(y)
    coeffs = pywt.wavedec(y, wavelet, mode='periodization')
    sigma_hat = estimate_sigma_mad(coeffs)
    lam = sigma_hat * np.sqrt(2.0 * np.log(n))
    n_detail = len(coeffs) - 1
    new_coeffs = [coeffs[0]]
    for level_idx, c in enumerate(coeffs[1:], start=1):
        if level_idx > n_detail - j0:
            new_coeffs.append(hard_threshold(c, lam))
        else:
            new_coeffs.append(c.copy())
    out = pywt.waverec(new_coeffs, wavelet, mode='periodization')
    return out[:n]


def mse(a: NDArray[np.float64], b: NDArray[np.float64]) -> float:
    return float(np.mean((a - b) ** 2))


rows = []
for name in ['Blocks', 'Bumps', 'HeaviSine', 'Doppler']:
    f_clean = signals[name]['clean']
    y = signals[name]['noisy']
    sigma = signals[name]['sigma']
    err_noisy = mse(y, f_clean)
    err_ideal = mse(ideal_wavelet_oracle(y, f_clean, sigma), f_clean)
    err_risk = mse(riskshrink(y)[0], f_clean)
    err_visu = mse(visushrink(y)[0], f_clean)
    err_hard = mse(hard_visu(y), f_clean)
    rows.append((name, err_noisy, err_ideal, err_risk, err_visu, err_hard))

print(f'{"Signal":<10}{"Noisy":>10}{"Ideal":>10}{"RiskShr":>10}{"VisuShr":>10}{"HardUniv":>10}')
print('-' * 60)
for r in rows:
    print(f'{r[0]:<10}{r[1]:>10.4f}{r[2]:>10.4f}{r[3]:>10.4f}{r[4]:>10.4f}{r[5]:>10.4f}')

print()
print('Paper Table 3 (single realisation):')
print('  Blocks    1.047 / 0.097 / 0.395 / 0.874')
print('  Bumps     0.937 / 0.111 / 0.496 / 1.058')
print('  HeaviSine 1.008 / 0.028 / 0.059 / 0.076')
print('  Doppler   0.9998 / 0.042 / 0.152 / 0.324')
print('Differences come from random seed; ordering of methods should match.')

---

## Part 8: Validation against PyWavelets denoise builtin / 라이브러리 비교

`pywt.threshold(x, value, 'soft')`는 우리의 `soft_threshold`와 동일해야 함.


In [ ]:
x = np.linspace(-5, 5, 1001)
for kind in ('soft', 'hard'):
    ours = soft_threshold(x, 1.5) if kind == 'soft' else hard_threshold(x, 1.5)
    theirs = pywt.threshold(x, 1.5, kind)
    diff = float(np.max(np.abs(ours - theirs)))
    print(f'{kind:>5s} max diff vs pywt.threshold: {diff:.2e}  (should be 0)')

---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 | Modern Equivalent / 현대 동등물 |
|---|---|---|---|
| Test functions | Table 1 (Blocks, Bumps, HeaviSine, Doppler) | `blocks`, `bumps`, `heavisine`, `doppler` | `pywt.data.demo_signal` (analogues) |
| DWT (Symmlet 8) | §1.4 \$\\mathcal W\$ | `pywt.wavedec(..., 'sym8')` | `pywt.swt`, `pywt.dwt2`, etc. |
| Hard threshold | Eq. (11) | `hard_threshold` | `pywt.threshold(x, λ, 'hard')` |
| Soft threshold | Eq. (12) | `soft_threshold` | `pywt.threshold(x, λ, 'soft')` |
| MAD noise estimation | §4.2 \$\\hat\\sigma = \\mathrm{MAD}/0.6745\$ | `estimate_sigma_mad` | `skimage.restoration.estimate_sigma` |
| Universal threshold | \$\\sigma\\sqrt{2\\log n}\$ | `lam = sigma*sqrt(2*log(n))` | (built-in across denoising libs) |
| Minimax threshold | \$\\lambda^*\_n\$ Table 2 | `riskshrink_threshold_grid` | `skimage.restoration.denoise_wavelet` (mode='soft') |
| VisuShrink | Definition 2 | `visushrink` | `skimage.restoration.denoise_wavelet(method='VisuShrink')` |
| RiskShrink | Definition 1 | `riskshrink` | (no direct equiv, more often replaced by SureShrink) |
| Projection oracle | §1.4 ideal SW | `ideal_wavelet_oracle` | (only for benchmarking) |

### Connections to next papers / 다음 논문과의 연결

- **Paper #2 SureShrink (Donoho-Johnstone 1995)** — replaces the *single* universal threshold with a *level-dependent* SURE-optimal threshold per dyadic band. Same `riskshrink`/`visushrink` skeleton, but `lam` is computed by minimising Stein's Unbiased Risk Estimate per level.
- **Paper #3 BayesShrink (Chang+ 2000)** — Laplace prior on coefficients, threshold becomes \$\\lambda\_{\\rm BS} = \\hat\\sigma^2 / \\hat\\sigma\_X\$ where \$\\hat\\sigma\_X\$ is per-band signal std.
- **Paper #7 BM3D** — abandons single-coefficient thresholding for *block-matched 3D groups* + collaborative filtering; soft/hard thresholding is reused inside the 3D transform domain.
- **Modern deep denoisers** (DnCNN, Restormer) implicitly learn shrinkage via residual learning; `nn.Softshrink` is the direct PyTorch analog of `soft_threshold`.

### Take-away / 결론

본 노트북은 \$n = 2048\$ 4개 시험 함수에 대해 RiskShrink, VisuShrink, hard-universal, ideal-wavelet oracle을 직접 구현하고, paper Table 3와 메소드 순서가 일치함을 확인했다. MAD 잡음 추정 정확도는 \$\\hat\\sigma\\approx 1\$로 양호. 핵심 식 (13) 오라클 부등식은 *implementation 차원에서* 시각적·수치적으로 다음을 확인시킨다: VisuShrink는 시각적으로 깨끗하지만 MSE는 RiskShrink보다 높다 — 이는 paper의 trade-off 분석과 정확히 일치한다.

Implements all four test functions, RiskShrink, VisuShrink, hard-universal, and the projection oracle. Method ordering matches Table 3. MAD-based \$\\hat\\sigma\\approx 1\$ is accurate. The visual–MSE trade-off (VisuShrink visually cleaner but higher MSE than RiskShrink) is reproduced exactly as the paper predicts.
